# tabular_ehr_baselines (full-record, v2) - logistic regression / decision tree / random forest
Full-record baselines (NO landmarks) that read what `tabular_data_prep` already built, so they are directly comparable with the full ProtoPregNet model.
- **Data:** the `tabular_v2` sparse matrix tables + the GLOBAL selected feature set (`selected_final = true`, chosen once on TRAIN rows by `tabular_data_prep`, so no selection happens here).
  Design = presence columns (sparse structure) + mean-imputed value columns, no censoring (the whole record is used).
- **Split:** the ProtoPregNet 8-1-1 split read from `protopregnet_landmark_cohort`. Optuna objective = validation macro AUROC, final model fit on TRAIN only, test touched once.
- **Optuna:** same set-up as `07_rf_landmark_eval_v2` (TPE, 20 random startup trials, no pruner, journal on local disk so the study is resumable, parallel trials).
  RF: identical space. DT: RF space minus the forest-only parameters. LR: only `C` and `class_weight` are tuned.
- **Jobs:** ONE job per model (`model` widget = rf | dt | lr); each job tunes, fits, predicts on test, computes metrics + bootstrap CIs and writes its own pickle. `mode=merge` combines the three pickles into `tabular_ehr_test_predictions.pkl`.
- **Joint multi-output:** DT / RF are natively joint (labels are multi-hot). LR has no joint form: three independent LRs per trial that share ONE hyperparameter set, scored on validation macro AUROC.

**Widgets (`mode`)**: `prep` (cache the v2 matrix on local disk; skipped if the cache already matches) | `run` (one model, set `model`) | `merge` (combine the per-model pickles).
**Parallelism on one CPU node:** a job uses `n_jobs` threads; if several jobs share the node keep (concurrent jobs x n_jobs) <= cores. `parallel_trials = 0` uses the per-model default in `CFG`.
Run `tabular_data_prep` first.

In [ ]:
import os, time, json, gc, shutil, pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, balanced_accuracy_score)

dbutils.widgets.dropdown("mode", "prep", ["prep", "run", "merge"])      # default prep: idempotent, so "run all" cannot start a long job by accident
dbutils.widgets.dropdown("model", "rf", ["rf", "dt", "lr"])
dbutils.widgets.text("n_jobs", "8")
dbutils.widgets.text("n_trials", "100")
dbutils.widgets.text("parallel_trials", "0")                            # 0 = per-model default from CFG
dbutils.widgets.dropdown("force", "false", ["false", "true"])
MODE = dbutils.widgets.get("mode")
MODEL = dbutils.widgets.get("model")
N_JOBS = int(dbutils.widgets.get("n_jobs"))
N_TRIALS = int(dbutils.widgets.get("n_trials"))
PARALLEL_TRIALS_WIDGET = int(dbutils.widgets.get("parallel_trials"))
FORCE = dbutils.widgets.get("force") == "true"

PERSON_ID_COL = "person_id"
CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]
LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]          # label definitions stored with the ProtoPregNet split table


def _pick_local_base():
    """Node-local scratch (matrix cache, Optuna journal): /local_disk0 on Databricks nodes, /tmp otherwise."""
    for base in ("/local_disk0", "/tmp"):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            return base
    return "/tmp"


EXPORT_DIR = "/path/to/exports"

CFG = {
    # ---- inputs: written by tabular_data_prep, plus the ProtoPregNet split ----
    "sparse_long_table": "catalog.schema.tabular_v2_sparse_long",
    "person_index_table": "catalog.schema.tabular_v2_person_index",
    "feature_index_table": "catalog.schema.tabular_v2_feature_index",
    "selected_table": "catalog.schema.tabular_v2_selected_features",              # GLOBAL selected set (selected_final = true); no selection is rerun here
    "tabular_meta_path": "/path/to/exports/tabular_v2/meta.json",  # matrix fingerprint
    "split_table": "catalog.schema.protopregnet_landmark_cohort",                  # EXACT ProtoPregNet 8-1-1 split (seed 42) + labels
    "cache_dir": "/path/to/exports/tabular_v2_cache",              # SAME cache as the RF / XGBoost landmark notebooks (reused, not rebuilt if the fingerprint matches)
    "local_cache_dir": f"{_pick_local_base()}/tabular_v2_cache",

    # ---- outputs ----
    "model_pickle_template": f"{EXPORT_DIR}/tabular_ehr_full_record_{{model}}.pkl",   # one per model job
    "merged_pickle_path": f"{EXPORT_DIR}/tabular_ehr_test_predictions.pkl",           # overwrites the previous export, as requested
    "pickle_source": "tabular_ehr_baselines_full_record",
    "hpo_dir": f"{EXPORT_DIR}/tabular_ehr_full_record_hpo",                           # best_params.json / trials.csv per model (durable copy)
    "hpo_local_dir": f"{_pick_local_base()}/tabular_ehr_full_record_hpo",             # Optuna journals live on local disk (safe for parallel trials)
    "model_order": ["lr", "dt", "rf"],
    "display_names": {"lr": "Tabular Logistic Regression", "dt": "Tabular Decision Tree", "rf": "Tabular Random Forest"},   # keys of payload["predictions"]
    "n_jobs": N_JOBS,
    "report_val": True,        # print validation macro AUROC of the final fit (it only chose the hyperparameters; the model is not fit on validation)

    # ---- per-model: parameters that are NOT tuned, the Optuna space, default concurrent trials ----
    # space entry: kind float | int | cat; 'when' = (parent, value): only suggested if the parent parameter has that value
    "models": {
        "rf": {
            "fixed": {"n_estimators": 300, "random_state": 42, "oob_score": False, "verbose": 0},   # more trees never hurt: a cost cap, not a tuned value (same as RF v2)
            "space": {
                "criterion":         {"kind": "cat", "choices": ["gini", "entropy"]},
                "max_depth":         {"kind": "cat", "choices": [None, 8, 12, 16, 24, 32]},
                "min_samples_leaf":  {"kind": "int", "low": 1, "high": 50, "log": True},
                "min_samples_split": {"kind": "int", "low": 2, "high": 40, "log": True},
                "max_features":      {"kind": "float", "low": 0.02, "high": 0.5, "log": True},
                "class_weight":      {"kind": "cat", "choices": [None, "balanced"]},
                "bootstrap":         {"kind": "cat", "choices": [True, False]},
                "max_samples":       {"kind": "float", "low": 0.3, "high": 1.0, "when": ("bootstrap", True)},
            },
            "parallel_trials": 4,      # each trial uses n_jobs // parallel_trials threads
        },
        "dt": {
            "fixed": {"random_state": 42},
            "space": {                  # the RF space minus the forest-only parameters (bootstrap, max_samples); splitter stays at its default
                "criterion":         {"kind": "cat", "choices": ["gini", "entropy"]},
                "max_depth":         {"kind": "cat", "choices": [None, 8, 12, 16, 24, 32]},
                "min_samples_leaf":  {"kind": "int", "low": 1, "high": 50, "log": True},
                "min_samples_split": {"kind": "int", "low": 2, "high": 40, "log": True},
                "max_features":      {"kind": "float", "low": 0.02, "high": 0.5, "log": True},
                "class_weight":      {"kind": "cat", "choices": [None, "balanced"]},
            },
            "parallel_trials": 8,      # a single tree fit is single-threaded, so concurrency comes from running trials side by side
        },
        "lr": {
            "fixed": {"penalty": "l2", "solver": "lbfgs", "max_iter": 1000, "tol": 1e-4, "random_state": 42},   # penalty is NOT tuned (feature selection already applied L1); only C and class_weight are
            "space": {
                "C":            {"kind": "float", "low": 1e-4, "high": 1e2, "log": True},
                "class_weight": {"kind": "cat", "choices": [None, "balanced"]},
            },
            "parallel_trials": 2,      # each trial fits 3 label-wise LRs in threads, so it wants >= 3 threads (8 // 2 = 4)
        },
    },
    "scale_value_columns": {"lr": True, "dt": False, "rf": False},   # standardize VALUE columns (train mean/std); presence columns stay 0/1. Trees are invariant to it.

    # ---- Optuna (same as 07_rf_landmark_eval_v2) ----
    "hpo": {
        "n_trials": N_TRIALS,
        "seed": 42,                    # TPE seed; with parallel trials the proposal order is not reproducible
        "n_startup_trials": 20,        # random trials before TPE starts modelling
    },

    # ---- bootstrap CIs on the test set (identical to the earlier notebooks) ----
    "n_bootstrap": 1000, "bootstrap_chunks": 8, "ci": 0.95, "bootstrap_seed": 42,
}
FULL_SPARSE_LONG_TABLE = CFG["sparse_long_table"]
PERSON_INDEX_TABLE = CFG["person_index_table"]
FEATURE_INDEX_TABLE = CFG["feature_index_table"]
print(f"mode={MODE} | model={MODEL} | n_jobs={N_JOBS} | sklearn {sklearn.__version__} | numpy {np.__version__}")

In [ ]:
# ===== cache I/O + shared design code: copied from 07_rf_landmark_eval_v2 (only what a full-record design needs) =====
CACHE_FILES = ["X_data.npy", "X_indices.npy", "X_indptr.npy", "person_index.npy", "feature_names.json", "meta.json"]


def _cp_to_volume(local_path, volume_path):
    """Volumes (FUSE) accept only sequential writes, so files are copied in from local disk instead of being written in place."""
    try:
        dbutils.fs.cp(f"file:{local_path}", volume_path)
    except Exception:
        shutil.copyfile(local_path, volume_path)


def _cp_from_volume(volume_path, local_path):
    try:
        dbutils.fs.cp(volume_path, f"file:{local_path}")
    except Exception:
        shutil.copyfile(volume_path, local_path)


def _mkdir_volume(path):
    try:
        os.makedirs(path, exist_ok=True)
    except OSError:
        dbutils.fs.mkdirs(path)


def read_meta(directory):
    with open(f"{directory}/meta.json") as f:
        return json.load(f)


def save_cache(X, feature_names, person_index, fingerprint):
    """Write the matrix once so every job loads it in seconds instead of re-reading ~680M Delta rows. Local disk first, then a verified sequential copy to the Volume."""
    X = X.tocsr()
    loc, tmp = CFG["local_cache_dir"], CFG["local_cache_dir"] + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    np.save(f"{tmp}/X_data.npy", X.data); np.save(f"{tmp}/X_indices.npy", X.indices); np.save(f"{tmp}/X_indptr.npy", X.indptr)
    np.save(f"{tmp}/person_index.npy", np.asarray(person_index).astype("int64"))
    with open(f"{tmp}/feature_names.json", "w") as f:
        json.dump(list(feature_names), f)
    with open(f"{tmp}/meta.json", "w") as f:                          # written last
        json.dump({"shape": list(X.shape), "nnz": int(X.nnz), "matrix_version": "tabular_v2", "fingerprint": fingerprint,
                   "created": time.strftime("%Y-%m-%d %H:%M:%S"), "sklearn": sklearn.__version__, "numpy": np.__version__}, f)
    shutil.rmtree(loc, ignore_errors=True); os.rename(tmp, loc)       # local copy becomes visible all at once
    _mkdir_volume(CFG["cache_dir"])
    for fn in CACHE_FILES[:-1] + ["meta.json"]:                       # meta.json last on the Volume too
        _cp_to_volume(f"{loc}/{fn}", f"{CFG['cache_dir']}/{fn}")
    for fn in CACHE_FILES:                                            # a partial copy must fail loudly here, not at load time
        a, b = os.path.getsize(f"{loc}/{fn}"), os.path.getsize(f"{CFG['cache_dir']}/{fn}")
        if a != b:
            raise IOError(f"cache copy to the Volume is incomplete: {fn} is {b} bytes there vs {a} bytes locally")
    print(f"cache written and verified: local {loc} and Volume {CFG['cache_dir']}")


def ensure_local_cache():
    """The matrix lives on the node's local disk so concurrent jobs can memory-map ONE copy (shared page cache). If the node has no local copy
    (e.g. after a cluster restart) it is copied from the Volume once; a task that loses the race just uses the winner's copy."""
    loc = CFG["local_cache_dir"]
    if os.path.exists(f"{loc}/meta.json"):
        return loc
    if not os.path.exists(f"{CFG['cache_dir']}/meta.json"):
        raise FileNotFoundError(f"no matrix cache in {CFG['cache_dir']} -- run this notebook once with mode=prep first")
    tmp = f"{loc}.tmp_{os.getpid()}"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for fn in CACHE_FILES:
        _cp_from_volume(f"{CFG['cache_dir']}/{fn}", f"{tmp}/{fn}")
    try:
        os.rename(tmp, loc)
    except OSError:
        shutil.rmtree(tmp, ignore_errors=True)                        # another task finished first
    return loc


def load_cache():
    loc = ensure_local_cache()
    with open(f"{loc}/meta.json") as f:
        meta = json.load(f)
    data, indices = np.load(f"{loc}/X_data.npy", mmap_mode="r"), np.load(f"{loc}/X_indices.npy", mmap_mode="r")
    X = sp.csr_matrix((data, indices, np.load(f"{loc}/X_indptr.npy")), shape=tuple(meta["shape"]), copy=False)   # read-only, memory-mapped
    with open(f"{loc}/feature_names.json") as f:
        feature_names = json.load(f)
    return X, feature_names, np.load(f"{loc}/person_index.npy")


def load_sparse_matrix_from_delta(persons_per_chunk=500):
    """Chunks strictly by PATIENT COUNT (never by an nnz-sized number, which once collapsed to a single unbounded load and killed the driver)."""
    person_meta = spark.table(PERSON_INDEX_TABLE).toPandas().sort_values("row_idx")
    feature_meta = spark.table(FEATURE_INDEX_TABLE).toPandas().sort_values("col_idx")
    person_index = person_meta["person_id"].tolist()
    feature_names = feature_meta["feature_name"].tolist()
    n_rows, n_cols = len(person_index), len(feature_names)
    total_nnz = spark.sql(f"SELECT COUNT(*) as n FROM {FULL_SPARSE_LONG_TABLE}").collect()[0]["n"]
    print(f"Loading {n_rows:,} x {n_cols:,} matrix, {total_nnz:,} nonzero...")
    rows_arr = np.empty(total_nnz, dtype=np.int32); cols_arr = np.empty(total_nnz, dtype=np.int32); vals_arr = np.empty(total_nnz, dtype=np.float32)
    pos = 0
    chunk_starts = list(range(0, n_rows, persons_per_chunk))
    assert len(chunk_starts) >= 2, f"only {len(chunk_starts)} chunk(s) for {n_rows:,} patients -- refusing to run unchunked"
    for i, start in enumerate(chunk_starts):
        end = min(start + persons_per_chunk, n_rows)
        chunk_df = spark.sql(f"SELECT row_idx, col_idx, value FROM {FULL_SPARSE_LONG_TABLE} WHERE row_idx >= {start} AND row_idx < {end}").toPandas()
        n = len(chunk_df)
        rows_arr[pos:pos + n] = chunk_df["row_idx"].values; cols_arr[pos:pos + n] = chunk_df["col_idx"].values; vals_arr[pos:pos + n] = chunk_df["value"].values
        pos += n
        if (i + 1) % 20 == 0 or i + 1 == len(chunk_starts):
            print(f"  chunk {i + 1}/{len(chunk_starts)} loaded")
        del chunk_df; gc.collect()
    if pos != total_nnz:
        print(f"  NOTE: filled {pos:,} of {total_nnz:,} allocated -- trimming buffers.")
        rows_arr, cols_arr, vals_arr = rows_arr[:pos], cols_arr[:pos], vals_arr[:pos]
    X_sparse = sp.csr_matrix((vals_arr, (rows_arr, cols_arr)), shape=(n_rows, n_cols))
    print(f"Loaded: {X_sparse.shape}, {X_sparse.nnz:,} nonzero.")
    return X_sparse, feature_names, pd.Index(person_index, name=PERSON_ID_COL)


def make_design_dense(Xc, base_idx, is_presence, impute):
    """Dense float32 design for the SELECTED design columns, in the order given.
      presence column: 1 if the stored base column has an entry (sparse STRUCTURE, so a week-0 first occurrence stays present), else 0
      value column:    the stored value if present, else impute (train mean of the present values, from the selected-features table)"""
    n = Xc.shape[0]
    sub = Xc[:, np.asarray(base_idx)].tocoo()
    out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
    out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
    return out


def matrix_fingerprint(feature_names, shape):
    """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
    import hashlib
    h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
    return h[:16]


def build_design_spec(sel, feature_names):
    """sel: the selected rows of tabular_v2_selected_features. Restricts the stored matrix to the base columns needed and maps each design column to its (restricted) base column."""
    col_index = {n: i for i, n in enumerate(feature_names)}
    missing = [c for c in sel["base_col"] if c not in col_index]
    assert not missing, f"{len(missing)} selected base columns are not in the cached matrix, e.g. {missing[:3]}: rebuild the cache (mode=prep)"
    base_full = sel["base_col"].map(col_index).to_numpy()
    uniq = np.unique(base_full)
    return {"uniq": uniq, "base_idx": np.searchsorted(uniq, base_full), "is_presence": (sel["role"] == "presence").to_numpy(),
            "impute": sel["impute_value"].to_numpy(dtype=float), "design_cols": sel["design_col"].tolist()}


# ---- pickle writer: local disk first, then a sequential copy (Volumes reject non-sequential writes); an existing file is replaced ----
def write_pickle(obj, volume_path):
    os.makedirs(CFG["hpo_local_dir"], exist_ok=True)
    local = f"{CFG['hpo_local_dir']}/_{os.path.basename(volume_path)}"
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    _mkdir_volume(os.path.dirname(volume_path))
    if os.path.exists(volume_path):
        os.remove(volume_path)
    _cp_to_volume(local, volume_path)
    if os.path.getsize(local) != os.path.getsize(volume_path):
        raise IOError(f"pickle copy to {volume_path} is incomplete")


def publish_text(text, volume_path):
    os.makedirs(CFG["hpo_local_dir"], exist_ok=True)
    local = f"{CFG['hpo_local_dir']}/_publish_{os.path.basename(volume_path)}"
    with open(local, "w") as f:
        f.write(text)
    _mkdir_volume(os.path.dirname(volume_path))
    _cp_to_volume(local, volume_path)

In [ ]:
# ============================================================================
# mode = prep  : cache the v2 matrix on local disk (skipped if the cache already matches the matrix fingerprint)
# mode = merge : combine the per-model pickles into ONE pickle (needs no matrix, no Spark)
# mode = run   : fall through; skipped here if this model's pickle already exists (force=true overwrites)
# ============================================================================
if MODE == "prep":
    t_prep = time.time()
    with open(CFG["tabular_meta_path"]) as f:
        tmeta = json.load(f)                                           # written by tabular_data_prep
    if os.path.exists(f"{CFG['cache_dir']}/meta.json") and not FORCE and read_meta(CFG["cache_dir"]).get("fingerprint") == tmeta["fingerprint"]:
        ensure_local_cache()                                           # refresh this node's local copy (e.g. after a cluster restart)
        msg = "cache already present for this matrix; nothing to rebuild (force=true rebuilds)"
        print(msg); dbutils.notebook.exit("prep done: " + msg)
    X, feature_names, person_index = load_sparse_matrix_from_delta()
    fp = matrix_fingerprint(feature_names, X.shape)
    assert fp == tmeta["fingerprint"], "the matrix tables do not match tabular_v2/meta.json: tabular_data_prep must be rerun (or the tables changed since)"
    save_cache(X, feature_names, person_index, fp)
    X_chk, names_chk, _ = load_cache()                                 # smoke test: the cache must load exactly what was saved
    assert X_chk.shape == X.shape and names_chk == feature_names, "cache reload does not match what was saved"
    print(f"cached {X.shape} matrix, {X.nnz:,} entries, fingerprint {fp}; prep took {time.time() - t_prep:.0f}s")
    dbutils.notebook.exit("prep done")

if MODE == "merge":
    parts = {}
    for m in CFG["model_order"]:
        path = CFG["model_pickle_template"].format(model=m)
        assert os.path.exists(path), f"{path} missing: run mode=run with model={m} first"
        with open(path, "rb") as f:
            parts[m] = pickle.load(f)
    ref = parts[CFG["model_order"][0]]
    for m, p in parts.items():                                         # every model must have been scored on the SAME test patients in the SAME order, from the same matrix
        assert np.array_equal(p["person_id"], ref["person_id"]), f"{m}: test patient order differs from {CFG['model_order'][0]}"
        assert np.array_equal(p["Y_true"], ref["Y_true"]), f"{m}: Y_true differs"
        assert p["matrix_fingerprint"] == ref["matrix_fingerprint"], f"{m}: built from a different matrix"
    merged = {k: ref[k] for k in ("source", "class_names", "person_id", "Y_true", "matrix_fingerprint", "split_table")}
    for k in ("predictions", "metrics", "ci", "best_params", "fixed_params", "study", "n_design_columns"):   # per-model dicts keyed by display name
        merged[k] = {}
        for m in CFG["model_order"]:
            merged[k].update(parts[m][k])
    merged["created"] = time.strftime("%Y-%m-%d %H:%M:%S")
    write_pickle(merged, CFG["merged_pickle_path"])
    print(f"merged {list(merged['predictions'])} on {len(merged['person_id']):,} test patients -> {CFG['merged_pickle_path']}")
    for name in merged["predictions"]:
        print(f"  {name:30s} test macro AUROC {merged['metrics'][name]['auroc_macro']:.4f} "
              f"[{merged['ci'][name]['auroc_macro'][1]:.4f}, {merged['ci'][name]['auroc_macro'][2]:.4f}]")
    dbutils.notebook.exit("merge done")

assert MODE == "run"
MODEL_PICKLE = CFG["model_pickle_template"].format(model=MODEL)
if not FORCE and os.path.exists(MODEL_PICKLE):
    dbutils.notebook.exit(f"{MODEL} already done: {MODEL_PICKLE} exists (set force=true to overwrite)")

In [ ]:
# ============================================================================
# mode = run : data. v2 cache + fingerprint check, the GLOBAL selected feature spec, the ProtoPregNet split + labels.
# FULL RECORD: every patient in the matrix, no eligibility filter, no censoring.
# ============================================================================
T = {}
def stage(name, t0): T[name] = round(time.time() - t0, 1)

t0 = time.time()
X_full, feature_names, person_index = load_cache()
pid_int_all = np.asarray(person_index).astype("int64")
with open(CFG["tabular_meta_path"]) as f:
    tmeta = json.load(f)
FP = matrix_fingerprint(feature_names, X_full.shape)
assert FP == tmeta["fingerprint"] == read_meta(CFG["local_cache_dir"]).get("fingerprint"), \
    "cache / tabular_v2 meta.json / matrix disagree: rerun mode=prep (and tabular_data_prep if the matrix changed)"

sel = (spark.table(CFG["selected_table"]).filter("selected_final = true")
       .select("design_col", "role", "base_col", "impute_value", "matrix_fingerprint").toPandas())
assert len(sel) > 0, f"no selected features in {CFG['selected_table']}"
assert set(sel["matrix_fingerprint"]) == {FP}, "the selected-features table was built from a different matrix: rerun tabular_data_prep"
sel = sel.sort_values(["role", "design_col"]).reset_index(drop=True)             # deterministic column order (same as RF v2)
spec = build_design_spec(sel, feature_names)
X = X_full[:, spec["uniq"]].tocsr()                                              # only the stored columns the selected design needs
IS_PRESENCE = spec["is_presence"]


def design(rows):
    """Dense float32 FULL-RECORD design (no censoring) for the given matrix rows."""
    return make_design_dense(X[rows], spec["base_idx"], IS_PRESENCE, spec["impute"])


coh = spark.table(CFG["split_table"]).select("pid_int", "split", *LABEL_COLS).toPandas().set_index("pid_int")
meta = coh.reindex(pid_int_all)
assert meta["split"].notna().all(), f"{int(meta['split'].isna().sum())} matrix patients are missing from {CFG['split_table']}"
Y_all = meta[LABEL_COLS].to_numpy(dtype=int)
split_all = meta["split"].to_numpy()
tr, va, te = (np.flatnonzero(split_all == s) for s in ("train", "val", "test"))
stage("load", t0)
print(f"global feature set: {len(sel):,} design columns ({int(IS_PRESENCE.sum()):,} presence + {int((~IS_PRESENCE).sum()):,} value) from {len(spec['uniq']):,} stored columns | fingerprint {FP}")
print(f"matrix {X_full.shape} | split sizes train {len(tr):,} / val {len(va):,} / test {len(te):,} | train prevalence {Y_all[tr].mean(0).round(3).tolist()} (control, pe_ghtn, gdm)")

In [ ]:
# ===== metrics + bootstrap CIs: identical to 07_rf_landmark_eval_v2 =====
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
        results[f"{m}_macro"] = macro(m)
    for m in ["auroc", "auprc"]:
        vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
        results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
    results["_primary"] = results["auroc_macro"]
    return results


def _boot_chunk(Y_true, Y_prob, class_names, n_iter, seed, keys):
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    out = {k: [] for k in keys}
    for _ in range(n_iter):
        idx = rng.randint(0, n, n)
        yt_r, yp_r = Y_true[idx], Y_prob[idx]
        if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names))):
            continue
        m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
        for k in keys:
            out[k].append(m[k])
    return out


def bootstrap_metrics_ci(Y_true, Y_prob, class_names):
    """Resample test patients, recompute the suite, percentile CI. Split into a FIXED number of chunks with fixed seeds so the result does not depend on n_jobs."""
    stems = ["auroc", "auprc", "f1", "bacc", "brier"]
    keys = ([f"{m}_macro" for m in stems] + ["auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
            + [f"{m}_{c}" for m in stems for c in class_names])
    sizes = np.full(CFG["bootstrap_chunks"], CFG["n_bootstrap"] // CFG["bootstrap_chunks"])
    sizes[: CFG["n_bootstrap"] % CFG["bootstrap_chunks"]] += 1
    parts = Parallel(n_jobs=min(CFG["n_jobs"], CFG["bootstrap_chunks"]))(
        delayed(_boot_chunk)(Y_true, Y_prob, class_names, int(s), CFG["bootstrap_seed"] + i, keys) for i, s in enumerate(sizes))
    lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
    res = {}
    for k in keys:
        arr = np.concatenate([np.asarray(p[k], dtype=float) for p in parts]); arr = arr[~np.isnan(arr)]
        res[k] = (float("nan"),) * 3 if len(arr) == 0 else (float(arr.mean()), *map(float, np.percentile(arr, [lo_q, hi_q])))
    return res

In [ ]:
# ============================================================================
# Model factory: one joint model per job. fit_model / predict_model are the ONLY places that know the model type.
#   rf / dt: natively joint multi-output (the three labels are multi-hot).
#   lr:      no joint form -> three independent LRs sharing ONE hyperparameter set (so one Optuna study, scored on validation macro AUROC).
# ============================================================================
def fit_value_scaler(Xtr, is_presence):
    """Train mean / std of the VALUE columns (presence columns stay 0/1). Zero-variance columns get std 1 so they become constant 0 instead of NaN."""
    val = ~is_presence
    mu = Xtr[:, val].mean(0, dtype=np.float64)
    sd = Xtr[:, val].std(0, dtype=np.float64)
    return {"val": val, "mu": mu.astype(np.float32), "sd": np.where(sd > 1e-12, sd, 1.0).astype(np.float32)}


def apply_value_scaler(Xd, sc):
    """IN PLACE (Xd is a fresh array from design(); avoids a second copy of the dense matrix)."""
    Xd[:, sc["val"]] -= sc["mu"]
    Xd[:, sc["val"]] /= sc["sd"]
    return Xd


def fit_model(model, Xtr, Ytr, hp, n_jobs):
    if any(int(Ytr[:, i].sum()) in (0, len(Ytr)) for i in range(Ytr.shape[1])):
        raise ValueError("a label has a single class in the training rows of this fit")
    params = {**CFG["models"][model]["fixed"], **hp}
    if model == "rf":
        return RandomForestClassifier(**params, n_jobs=int(n_jobs)).fit(Xtr, Ytr)
    if model == "dt":
        return DecisionTreeClassifier(**params).fit(Xtr, Ytr)
    one = lambda c: LogisticRegression(**params).fit(Xtr, Ytr[:, c])           # lr: one binary LR per label
    return Parallel(n_jobs=max(1, min(len(CLASS_NAMES), int(n_jobs))), backend="threading")(delayed(one)(c) for c in range(len(CLASS_NAMES)))


def predict_model(model, fitted, Xd):
    """(n, 3) probabilities, columns in CLASS_NAMES order."""
    if model == "lr":
        return np.column_stack([m.predict_proba(Xd)[:, 1] for m in fitted])
    return np.column_stack([p[:, 1] for p in fitted.predict_proba(Xd)])         # rf / dt: list of (n, 2), one per output

In [ ]:
# ============================================================================
# Optuna study driver: same set-up as 07_rf_landmark_eval_v2 (TPE seed 42, 20 random startup trials, no pruner, journal on local disk so the
# study is resumable, parallel trials sharing the node's cores, objective = validation macro AUROC; the test split is never used).
# ============================================================================
def make_journal_storage(optuna, path):
    """A journal file on local disk: safe for parallel trials, and an interrupted study resumes from it. API moved in Optuna 4."""
    try:
        from optuna.storages.journal import JournalFileBackend                       # Optuna >= 4.0
        return optuna.storages.JournalStorage(JournalFileBackend(path))
    except ImportError:
        return optuna.storages.JournalStorage(optuna.storages.JournalFileStorage(path))   # Optuna 3.x


def suggest_from_space(trial, space):
    """Each entry: {kind: float|int|cat, low, high, log, choices, when: (parent, value)}; a parameter with 'when' is only suggested if its parent has that value."""
    hp = {}
    for name, s in space.items():
        if "when" in s and hp.get(s["when"][0]) != s["when"][1]:
            continue
        if s["kind"] == "cat":
            hp[name] = trial.suggest_categorical(name, s["choices"])
        elif s["kind"] == "int":
            hp[name] = trial.suggest_int(name, s["low"], s["high"], log=s.get("log", False))
        else:
            hp[name] = trial.suggest_float(name, s["low"], s["high"], log=s.get("log", False))
    return hp


def run_study(model, Xtr_d, Ytr, Xva_d, Yva):
    try:
        import optuna
    except ImportError as e:
        raise ImportError("this notebook needs Optuna: add 'optuna==3.6.1' as a cluster library (Libraries tab) and restart the cluster") from e
    optuna.logging.set_verbosity(optuna.logging.WARNING)                             # our progress line replaces Optuna's per-trial log
    hpo, mc = CFG["hpo"], CFG["models"][model]
    study_name = f"{model}_full_record"
    parallel_trials = max(1, min(PARALLEL_TRIALS_WIDGET or mc["parallel_trials"], N_JOBS))
    threads = max(1, N_JOBS // parallel_trials)                                      # concurrent trials share the node's cores
    os.makedirs(CFG["hpo_local_dir"], exist_ok=True)
    journal = f"{CFG['hpo_local_dir']}/{study_name}.journal"
    if FORCE and os.path.exists(journal):
        os.remove(journal)                                                           # force=true starts the study from scratch
    study = optuna.create_study(
        study_name=study_name, storage=make_journal_storage(optuna, journal), direction="maximize", load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=hpo["seed"], n_startup_trials=hpo["n_startup_trials"]), pruner=optuna.pruners.NopPruner())
    finished = (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)
    done = sum(t.state in finished for t in study.trials)
    remaining = max(0, hpo["n_trials"] - done)
    print(f"study '{study_name}': {done} trials already finished, {remaining} to run, {parallel_trials} in parallel x {threads} threads | optuna {optuna.__version__}")

    def objective(trial):
        hp = suggest_from_space(trial, mc["space"])
        P = predict_model(model, fit_model(model, Xtr_d, Ytr, hp, threads), Xva_d)
        aucs = [float(roc_auc_score(Yva[:, i], P[:, i])) for i in range(len(CLASS_NAMES))]
        trial.set_user_attr("val_auc_by_label", dict(zip(CLASS_NAMES, aucs)))
        return float(np.mean(aucs))                                                  # validation macro AUROC (test is never touched)

    def save_snapshot(study_, final):
        complete = [t for t in study_.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not complete:
            return
        best = study_.best_trial
        payload = {"study_name": study_name, "model": model, "best_value": float(best.value), "best_trial": int(best.number), "best_params": dict(best.params),
                   "fixed_params": mc["fixed"], "val_auc_by_label": best.user_attrs.get("val_auc_by_label"), "n_trials_complete": len(complete),
                   "objective": "mean validation AUROC over control / pe_ghtn / gdm, full-record features (global selected set), train->val", "final": bool(final),
                   "matrix_fingerprint": FP, "optuna": optuna.__version__, "created": time.strftime("%Y-%m-%d %H:%M:%S")}
        publish_text(json.dumps(payload, indent=2, default=str), f"{CFG['hpo_dir']}/{model}/best_params.json")
        publish_text(study_.trials_dataframe().to_csv(index=False), f"{CFG['hpo_dir']}/{model}/trials.csv")

    def progress(study_, trial):
        n_fin = sum(t.state in finished for t in study_.trials)
        best = max((t.value for t in study_.trials if t.state == optuna.trial.TrialState.COMPLETE), default=float("nan"))
        print(f"trial {trial.number:4d} {trial.state.name:8s} value {trial.value if trial.value is not None else float('nan'):.4f} | best {best:.4f} | {n_fin}/{hpo['n_trials']}")
        if n_fin % 10 == 0:
            save_snapshot(study_, final=False)                                       # progress is visible (and safe) while the study runs

    t0 = time.time()
    if remaining:
        study.optimize(objective, n_trials=remaining, n_jobs=parallel_trials, callbacks=[progress], gc_after_trial=True)
    stage("tune", t0)
    save_snapshot(study, final=True)
    print(f"best validation macro AUROC {study.best_value:.4f} (trial {study.best_trial.number}) | tuning took {T['tune']}s")
    print("best params:", json.dumps(study.best_params, indent=1, default=str))
    return study

In [ ]:
# ============================================================================
# mode = run : tune on train -> val, fit the winner on TRAIN only, predict test, metrics + bootstrap CIs, pickle.
# ============================================================================
DISP = CFG["display_names"][MODEL]
Ytr, Yva, Yte = Y_all[tr], Y_all[va], Y_all[te]

t0 = time.time()
Xtr_d, Xva_d = design(tr), design(va)
scaler = None
if CFG["scale_value_columns"][MODEL]:
    scaler = fit_value_scaler(Xtr_d, IS_PRESENCE)                                    # statistics from TRAIN rows only
    apply_value_scaler(Xtr_d, scaler); apply_value_scaler(Xva_d, scaler)
stage("tune_data", t0)
print(f"{DISP}: tuning data train {Xtr_d.shape}, val {Xva_d.shape}{' (value columns standardized)' if scaler else ''}")

study = run_study(MODEL, Xtr_d, Ytr, Xva_d, Yva)
HP = dict(study.best_params)

t0 = time.time(); fitted = fit_model(MODEL, Xtr_d, Ytr, HP, N_JOBS); stage("fit", t0)         # final fit: TRAIN only, best config
if CFG["report_val"]:
    Pv = predict_model(MODEL, fitted, Xva_d)
    va_auc = [float(roc_auc_score(Yva[:, i], Pv[:, i])) for i in range(len(CLASS_NAMES)) if 0 < Yva[:, i].sum() < len(va)]
    print(f"validation macro AUROC {np.mean(va_auc):.4f} (not fit on validation; the study chose the hyperparameters with it)")
del Xtr_d, Xva_d; gc.collect()

t0 = time.time()
Xte_d = design(te)                                                                    # the test design is built only now, after tuning is finished
if scaler:
    apply_value_scaler(Xte_d, scaler)
p_te = predict_model(MODEL, fitted, Xte_d)
metrics = _compute_metric_suite(Yte, p_te)
stage("predict", t0)
t0 = time.time(); ci = bootstrap_metrics_ci(Yte, p_te, CLASS_NAMES); stage("bootstrap", t0)
print(f"{DISP}: test n={len(te):,} | macro AUROC {metrics['auroc_macro']:.4f} [{ci['auroc_macro'][1]:.4f}, {ci['auroc_macro'][2]:.4f}] | "
      + " ".join(f"{c} {metrics[f'auroc_{c}']:.4f}" for c in CLASS_NAMES))
for m_ in ["auroc", "auprc", "f1", "brier", "bacc"]:
    print(f"  {m_:6s} macro {metrics[f'{m_}_macro']:.4f}  [{ci[f'{m_}_macro'][1]:.4f}, {ci[f'{m_}_macro'][2]:.4f}]")

# ---- one pickle per model; same layout as the previous export (source / class_names / person_id / Y_true / predictions) plus the extras ----
payload = {
    "source": CFG["pickle_source"],
    "class_names": CLASS_NAMES,
    "person_id": np.asarray(person_index)[te].astype(str),          # test patients in matrix order; Y_true / predictions follow this order
    "Y_true": np.asarray(Yte),
    "predictions": {DISP: np.asarray(p_te)},
    "metrics": {DISP: metrics},                                      # full metric suite on the test set (macro, per-class, outcomes-only macro)
    "ci": {DISP: ci},                                                # bootstrap (mean, ci_lo, ci_hi) for every metric
    "best_params": {DISP: HP},
    "fixed_params": {DISP: CFG["models"][MODEL]["fixed"]},
    "study": {DISP: {"study_name": study.study_name, "best_value_val_macro_auroc": float(study.best_value), "best_trial": int(study.best_trial.number),
                     "val_auc_by_label": study.best_trial.user_attrs.get("val_auc_by_label"), "n_trials_requested": CFG["hpo"]["n_trials"],
                     "n_trials_complete": int(sum(t.state.name == "COMPLETE" for t in study.trials))}},
    "n_design_columns": {DISP: int(len(sel))},
    "matrix_fingerprint": FP,
    "split_table": CFG["split_table"],
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
}
write_pickle(payload, MODEL_PICKLE)
print(f"saved {MODEL_PICKLE} | stage seconds: {T}")